# HMARL Full Experiment Runner (Colab GPU)

This notebook is the **project-completion runner** for Google Colab. It is designed to:

1. mount Google Drive for persistent outputs,
2. clone the repo and install dependencies,
3. run a **main continuous experiment** on GPU,
4. run the **single-mission + ground-truth validation benchmark**, and
5. save reports, traces, plots, and model checkpoints back to Drive.

A practical note: for the current scope of this project, the codebase is ready for **end-to-end runs and final reporting**. What is still open scientifically is larger-scale realism work and broader multi-seed studies. So this notebook is enough to finish the current project deliverables, but it does not mean the research problem is permanently closed.

Because Colab often uses Python 3.10/3.11, this notebook installs from `requirements.txt` and runs directly from the repo checkout rather than relying on package installation from `pyproject.toml`.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

GIT_BRANCH = 'codex/fix-diagnostics-and-sync-updates'
REPO_ROOT = Path('/content/qgi_lab_hmarl')
RUNS_ROOT = Path('/content/drive/MyDrive/qgi_lab_hmarl_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Runs root:', RUNS_ROOT)


In [ ]:
import shutil
import subprocess
import sys

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/zhongnz/qgi_lab_hmarl.git', str(REPO_ROOT)],
    check=True,
)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all'], check=True)

checkout = subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', GIT_BRANCH])
if checkout.returncode != 0:
    print(f'Branch {GIT_BRANCH!r} not available remotely; falling back to main.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', 'main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)

print('Clone and install complete.')


In [ ]:
import os
import platform
import subprocess
import sys

import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('No GPU runtime detected. Switch Colab runtime to GPU before running the full experiment.')

subprocess.run(['bash', '-lc', 'nvidia-smi || true'], check=False)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE


In [ ]:
import json
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(REPO_ROOT))

from hmarl_mvp.config import get_default_config
from hmarl_mvp.experiment import run_trained_mappo_trace
from hmarl_mvp.mappo import MAPPOConfig, MAPPOTrainer
from hmarl_mvp.plotting import plot_time_series_diagnostics, plot_training_curves
from hmarl_mvp.report import generate_training_report


def run_colab_training(
    run_name: str,
    *,
    env_overrides: dict | None = None,
    mappo_overrides: dict | None = None,
    seed: int = 42,
    eval_interval: int = 10,
    eval_episodes: int = 5,
):
    """Run one end-to-end training/evaluation job and save full artifacts."""
    out_dir = RUNS_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    env_cfg = get_default_config(**(env_overrides or {}))
    mappo_kwargs = {
        'rollout_length': 64,
        'lr': 3e-4,
        'entropy_coeff': 0.01,
        'total_iterations': 100,
        'device': DEVICE,
    }
    if mappo_overrides:
        mappo_kwargs.update(mappo_overrides)
    mappo_cfg = MAPPOConfig(**mappo_kwargs)

    env_cfg['rollout_steps'] = max(int(env_cfg.get('rollout_steps', 20)), int(mappo_cfg.rollout_length) + 5)

    print(f'Running: {run_name}')
    print('  device:', mappo_cfg.device)
    print('  env:', {k: env_cfg[k] for k in ['num_vessels', 'num_ports', 'forecast_source', 'episode_mode', 'mission_success_on', 'rollout_steps']})
    print('  mappo:', {'rollout_length': mappo_cfg.rollout_length, 'total_iterations': mappo_cfg.total_iterations, 'lr': mappo_cfg.lr, 'entropy_coeff': mappo_cfg.entropy_coeff})

    t0 = time.time()
    trainer = MAPPOTrainer(env_config=env_cfg, mappo_config=mappo_cfg, seed=seed)
    history = trainer.train(
        num_iterations=int(mappo_cfg.total_iterations),
        eval_interval=eval_interval,
    )
    elapsed = time.time() - t0

    history_df = pd.DataFrame(history)
    history_df.to_csv(out_dir / 'train_history.csv', index=False)

    trainer.save_models(str(out_dir / 'final_model'))

    eval_result = trainer.evaluate_episodes(num_episodes=eval_episodes)
    with open(out_dir / 'eval_result.json', 'w') as f:
        json.dump(eval_result, f, indent=2, default=str)

    trace_result = run_trained_mappo_trace(
        trainer,
        num_steps=int(env_cfg['rollout_steps']),
        return_logs=True,
    )
    if isinstance(trace_result, tuple):
        eval_trace, eval_action_trace, eval_event_log = trace_result
    else:
        eval_trace = trace_result
        eval_action_trace = pd.DataFrame()
        eval_event_log = pd.DataFrame()

    eval_trace.to_csv(out_dir / 'eval_trace.csv', index=False)
    eval_action_trace.to_csv(out_dir / 'eval_action_trace.csv', index=False)
    eval_event_log.to_csv(out_dir / 'eval_event_log.csv', index=False)

    report = generate_training_report(
        history=history,
        eval_result=eval_result,
        config={'env': env_cfg, 'mappo': mappo_cfg.__dict__},
        elapsed_seconds=elapsed,
    )
    (out_dir / 'report.md').write_text(report)

    plot_training_curves(history_df, out_path=str(out_dir / 'training_curves.png'))
    plot_time_series_diagnostics(eval_trace, out_path=str(out_dir / 'diagnostics_trace.png'), column_group='aggregate')
    plot_time_series_diagnostics(eval_trace, out_path=str(out_dir / 'diagnostics_trace_vessels.png'), column_group='vessel')
    plot_time_series_diagnostics(eval_trace, out_path=str(out_dir / 'diagnostics_trace_ports.png'), column_group='port')
    plot_time_series_diagnostics(eval_trace, out_path=str(out_dir / 'diagnostics_trace_coordinators.png'), column_group='coordinator')

    summary = {
        'run_name': run_name,
        'out_dir': str(out_dir),
        'elapsed_seconds': elapsed,
        'eval_mean': eval_result['mean'],
    }
    with open(out_dir / 'run_summary.json', 'w') as f:
        json.dump(summary, f, indent=2, default=str)

    print(f"Finished {run_name} in {elapsed:.1f}s")
    print('Outputs:', out_dir)
    return summary


## Experiment presets

This notebook defines two runs:

- **Main continuous experiment**: the current operational task using the default continuous scheduler.
- **Validation benchmark**: `single_mission + ground_truth`, which is easier to interpret and useful for validating the core control loop.

For the current project scope, the main run is the one you would use for the final operating narrative. The validation benchmark is there to support the methodology and sanity-check the core system behavior.


In [ ]:
RUN_MAIN_EXPERIMENT = True
RUN_VALIDATION_BENCHMARK = True

MAIN_RUN_NAME = 'colab_main_continuous_final'
MAIN_ENV_OVERRIDES = {
    'episode_mode': 'continuous',
    'forecast_source': 'heuristic',
}
MAIN_MAPPO_OVERRIDES = {
    'rollout_length': 64,
    'total_iterations': 100,
    'device': DEVICE,
}

VALIDATION_RUN_NAME = 'colab_single_mission_ground_truth'
VALIDATION_ENV_OVERRIDES = {
    'episode_mode': 'single_mission',
    'mission_success_on': 'arrival',
    'forecast_source': 'ground_truth',
}
VALIDATION_MAPPO_OVERRIDES = {
    'rollout_length': 32,
    'total_iterations': 20,
    'device': DEVICE,
}

main_result = None
validation_result = None

print('Main run device:', DEVICE)
print('Main run name:', MAIN_RUN_NAME)
print('Validation run name:', VALIDATION_RUN_NAME)


In [ ]:
if RUN_MAIN_EXPERIMENT:
    main_result = run_colab_training(
        MAIN_RUN_NAME,
        env_overrides=MAIN_ENV_OVERRIDES,
        mappo_overrides=MAIN_MAPPO_OVERRIDES,
        seed=42,
        eval_interval=10,
        eval_episodes=5,
    )
    pd.DataFrame([main_result['eval_mean']])
else:
    print('Main experiment skipped.')


In [ ]:
if RUN_VALIDATION_BENCHMARK:
    validation_result = run_colab_training(
        VALIDATION_RUN_NAME,
        env_overrides=VALIDATION_ENV_OVERRIDES,
        mappo_overrides=VALIDATION_MAPPO_OVERRIDES,
        seed=42,
        eval_interval=5,
        eval_episodes=5,
    )
    pd.DataFrame([validation_result['eval_mean']])
else:
    print('Validation benchmark skipped.')


In [ ]:
comparison_rows = []
if main_result is not None:
    m = main_result['eval_mean']
    comparison_rows.append({
        'run': 'continuous_main',
        'total_reward': m.get('total_reward'),
        'on_time_rate': m.get('on_time_rate'),
        'completed_arrivals': m.get('completed_arrivals'),
        'total_vessels_served': m.get('total_vessels_served'),
        'mission_success_rate': m.get('mission_success_rate'),
        'total_ops_cost_usd': m.get('total_ops_cost_usd'),
    })
if validation_result is not None:
    v = validation_result['eval_mean']
    comparison_rows.append({
        'run': 'single_mission_ground_truth',
        'total_reward': v.get('total_reward'),
        'on_time_rate': v.get('on_time_rate'),
        'completed_arrivals': v.get('completed_arrivals'),
        'total_vessels_served': v.get('total_vessels_served'),
        'mission_success_rate': v.get('mission_success_rate'),
        'total_ops_cost_usd': v.get('total_ops_cost_usd'),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(RUNS_ROOT / 'colab_experiment_comparison.csv', index=False)
comparison_df


In [ ]:
print('Finished. Artifacts are in:', RUNS_ROOT)
print('Suggested deliverables to review:')
print(' - train_history.csv')
print(' - eval_result.json')
print(' - eval_trace.csv')
print(' - report.md')
print(' - training_curves.png')
print(' - diagnostics_trace*.png')
